In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

In [2]:
from verimon import loaders
from random import randrange
from math import sqrt

mc_sl_u_nxn = "../tests/snake_ladder/mc_u_nxn.pm"

def random_ladder(n):
    source = randrange(1,n-int(sqrt(n)))
    dest = randrange(source+int(sqrt(n)), int(min(n, source + n / 2)))
    return source, dest

def random_snake(n):
    source = randrange(int(sqrt(n))+2,n)
    dest = randrange(1, source-int(sqrt(n)))
    return source, dest

n = 4**2
ladders = {0:0}
snakes = {0:0}
while not set(ladders.keys()).isdisjoint(set(snakes.keys())):
    ladders = dict(random_ladder(n) for _ in range(int(sqrt(n))))
    snakes = dict(random_snake(n) for _ in range(int(sqrt(n))))

# Random snakes and ladders
mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n, ladders, snakes)

milton_snakes = {98: 76, 95: 75, 93: 73, 87: 24, 64: 60, 62: 19, 55: 53, 49: 11, 47: 26, 16: 6}
milton_ladders = {1: 38, 4: 14, 9: 31, 28: 64, 40: 42, 36: 44, 51: 67, 71: 91, 80: 100}
# mc = loaders.load_snl(mc_sl_u_nxn, n := 10**2, ladders:=milton_ladders, snakes:=milton_snakes)

In [3]:
from stormvogel.show import show
from stormvogel.mapping import stormpy_to_stormvogel

mc_sv = stormpy_to_stormvogel(mc)
loaders._add_valuation_to_sv_labels(mc, mc_sv)
show(mc_sv)

Output()

Output()

In [4]:
from verimon.MonitorLearning import learn_monitor

learned_monitor = learn_monitor(
    mc, 
    "init", 
    ["init", "normal", "snake", "ladder"], 
    'P>0.9 [ "good" ]', 
    0.7, 
    walks_per_state= 1000, 
    walk_len=100
)

Hypothesis 1: 1 states.
Hypothesis 2: 6 states.
Hypothesis 3: 10 states.
Hypothesis 4: 12 states.
Hypothesis 5: 14 states.
Hypothesis 6: 15 states.
Hypothesis 7: 16 states.
Hypothesis 8: 18 states.
-----------------------------------
Learning Finished.
Learning Rounds:  8
Number of states: 18
Time (in seconds)
  Total                : 4.73
  Learning algorithm   : 0.04
  Conformance checking : 4.69
Learning Algorithm
 # Membership Queries  : 399
 # MQ Saved by Caching : 223
 # Steps               : 3789
Equivalence Query
 # Membership Queries  : 18000
 # Steps               : 1040591
-----------------------------------


In [6]:
from verimon.MonitorLearning import aalpy_dfa_to_stormvogel
from verimon.unrolling import simulator_unroll, prune_monitor
from verimon.algs import complement_model

mon_cycl = aalpy_dfa_to_stormvogel(learned_monitor)
# show(mon_cycl)
complement_model(mon_cycl, "accepting")
mon = simulator_unroll(mon_cycl, 10)
prune_monitor(mon)

In [7]:
gb_e = "../tests/gb_exact.pm"
gb = loaders.load_dfa(gb_e)

In [16]:
from verimon.MDP_product import MC_MON_Product
from time import time

t = time()
pomdp = MC_MON_Product(mc_sv, mon, gb, "good")
# pomdp.apply_spec('P>=0.5 [ F<=3 "good"]')
pomdp.create_product(use_step_label=True)
print(f"Starting Paynt after {time() - t}s\n--------------------")

assignment = pomdp.check_paynt_prop('Pmax=? [ (F "goal") ]', 0.2)
# show(pomdp.pomdp)

Remove 3322 unreachable states
Starting Paynt after 53.36067271232605s
--------------------
2024-10-17 11:22:15,047 - pomdp.py - constructed POMDP having 16 observations.
2024-10-17 11:22:15,050 - pomdp.py - unfolding 1-FSC template into POMDP...
2024-10-17 11:22:15,052 - pomdp.py - constructed quotient MDP having 386 states and 1891 actions.
2024-10-17 11:22:15,054 - statistic.py - synthesis initiated, design space: 1e9
--------------------
Synthesis summary:
constraint 1: P>9/10 [F "goal"]

method: AR, synthesis time: 1.9 s
number of holes: 14, family size: 1e9, quotient: 386 states / 1891 actions
explored: 100 %
MDP stats: avg MDP size: 95, iterations: 706

feasible: no
--------------------
counterexample not found


In [1]:
# print(pomdp.find_trace_to_good_state())

NameError: name 'pomdp' is not defined

In [10]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

poss = pomdp.simulate_paynt_assignment(assignment, 1000)
player_path = poss
# player_path = [(0, [])]

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in pomdp.mc.states.values() 
                if "good" in state.labels]

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

s0, labels=!g0 init !s0 accepting [pos=0] !l0


it took 1000 tries until the goal was reached


<IPython.core.display.Javascript object>

In [17]:
from stormvogel.show import show
from stormvogel.mapping import stormpy_to_stormvogel
from stormpy import model_checking, parse_properties

induced_mc = pomdp.created_induced_mc(assignment)

imc =  stormpy_to_stormvogel(induced_mc)
result_goal = model_checking(induced_mc, parse_properties('Pmax=? [F "goal"]')[0])
result_stop = model_checking(induced_mc, parse_properties('Pmax=? [F "stop"]')[0])
prop_goal = result_goal.at(induced_mc.initial_states[0])
prop_stop = result_stop.at(induced_mc.initial_states[0])
print(f"probability to reach goal={prop_goal} and stop={prop_stop}. Total={prop_goal+prop_stop}")
show(imc)


AttributeError: 'NoneType' object has no attribute 'num_holes'